In [130]:
import os
import rasterio as rio
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.path as mpth
from pathlib import Path


import Functions
import importlib

importlib.reload(Functions)

<module 'Functions' from '/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/Functions/Functions.py'>

In [131]:
app_path = Functions.get_input_path() / 'App'
input_path = app_path / 'Documents' / 'csv'

path: /home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA


In [132]:
sigma_TM = pd.DataFrame(pd.read_csv(input_path / 'TimeSeries_sigma.csv'))
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
date_S1 = list(pd.unique(pd.to_datetime(sigma_TM['Date'])))
sigma_TM.drop(sigma_TM[sigma_TM['mean'] == 0].index,inplace=True)


sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date'])
sigma_TM['diff_mean'] = -sigma_TM.groupby(['Code', 'Band'])['mean'].diff(periods=-1)
sigma_TM['diff_lin'] = 10 ** (sigma_TM['diff_mean']/10)
display(sigma_TM)

,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,diff_lin
1298,1545384,20170104_VH,-15.490180,1.038437,2017-01-04,VH,DB_SB2_C,Potatoes,1.343919,1.362674
1300,1545384,20170110_VH,-14.146261,1.136036,2017-01-10,VH,DB_SB2_C,Potatoes,-6.476397,0.225092
1302,1545384,20170116_VH,-20.622658,0.922879,2017-01-16,VH,DB_SB2_C,Potatoes,-1.428043,0.719773
1304,1545384,20170122_VH,-22.050701,0.694627,2017-01-22,VH,DB_SB2_C,Potatoes,5.231783,3.335633
1306,1545384,20170128_VH,-16.818918,0.731224,2017-01-28,VH,DB_SB2_C,Potatoes,0.679199,1.169284
...,...,...,...,...,...,...,...,...,...,...
935,2278835,20171206_VV,-6.205426,0.920910,2017-12-06,VV,PVD-M-C,Corn,-5.569663,0.277354
937,2278835,20171212_VV,-11.775089,0.645342,2017-12-12,VV,PVD-M-C,Corn,5.853215,3.848766
939,2278835,20171218_VV,-5.921874,1.043583,2017-12-18,VV,PVD-M-C,Corn,-0.802364,0.831311
941,2278835,20171224_VV,-6.724238,1.070252,2017-12-24,VV,PVD-M-C,Corn,0.244181,1.057835


In [133]:
moisture = pd.read_excel('/home/frank/Desktop/f.chiapperino_local'
                        '/VALENCIA/SCUOLA/App/Data/Ground_Campaign/Flevoland_data/Data_25_fields/Average_Soil_moisture_N.xlsx',
                        header=0)

moisture.set_index('Code',inplace=True)
moisture = moisture[moisture.index.notna()]
moisture_date = list(pd.to_datetime(moisture.columns))

df_date_moist = pd.DataFrame({'original_date': moisture_date}).sort_values('original_date')
df_date_S1 = pd.DataFrame({'ref_date': date_S1}).sort_values('ref_date')

new_date_moist = pd.merge_asof(df_date_moist, df_date_S1, 
                                left_on='original_date',
                                right_on='ref_date', direction='backward')
moisture.columns = pd.to_datetime(new_date_moist['ref_date'])
display(type(moisture.columns[0]))

moisture = moisture.reset_index().melt(id_vars='Code', var_name='Date', value_name='Moist_situ')
display(np.unique(moisture['Code']))


pandas.Timestamp

array([1553694., 1576641., 1601502., 1631664., 1697689., 1697690.,
       1698168., 1824038., 1841223., 1841224., 1841225., 1896343.,
       1936133., 1936134., 1936135., 2041694., 2081267., 2081268.,
       2081887., 2278835.])

In [134]:
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
moisture['Date'] = pd.to_datetime(moisture['Date'])

Try the aplha method: $$SSM_{i+1} \approx \frac{\sigma_{0}^{i+1}}{\sigma_{0}^{i}} SSM_{i}$$

In [135]:
sigma_TM = pd.merge(
    sigma_TM, 
    moisture, 
    on=['Code', 'Date'], 
    how='right' 
)
sigma_TM = sigma_TM[sigma_TM['Date'] >= moisture['Date'].iloc[0]]


sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date']).reset_index(drop=True)
display(sigma_TM)


,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,diff_lin,Moist_situ
0,1553694,20170516_VH,-24.067930,1.918145,2017-05-16,VH,DB_SB2_C,Beets,1.608618,1.448311,19.1125
1,1553694,20170522_VH,-22.459312,1.242794,2017-05-22,VH,DB_SB2_C,Beets,2.084634,1.616082,16.2500
2,1553694,20170528_VH,-20.374678,1.630524,2017-05-28,VH,DB_SB2_C,Beets,2.685234,1.855767,NaN
3,1553694,20170603_VH,-17.689444,1.524553,2017-06-03,VH,DB_SB2_C,Beets,3.346001,2.160728,27.5125
4,1553694,20170609_VH,-14.343443,1.547611,2017-06-09,VH,DB_SB2_C,Beets,-1.950233,0.638229,15.9500
...,...,...,...,...,...,...,...,...,...,...,...
635,2278835,20170814_VV,-11.082822,0.921020,2017-08-14,VV,PVD-M-C,Corn,0.046305,1.010719,27.2500
636,2278835,20170820_VV,-11.036517,1.013179,2017-08-20,VV,PVD-M-C,Corn,-0.675594,0.855935,26.6500
637,2278835,20170901_VV,-11.232622,0.884906,2017-09-01,VV,PVD-M-C,Corn,0.269737,1.064079,NaN
638,2278835,20170907_VV,-10.962885,0.753886,2017-09-07,VV,PVD-M-C,Corn,0.013563,1.003128,26.9250


In [136]:
def run_recursion_numpy(ssm_arr, ratio_arr, win=4):

    ssm_ret = ssm_arr.copy().astype(float)
    
    for b in range(0, len(ssm_ret), win):
        for passo in range(1, win):
    
            i = b + passo
            if i >= len(ssm_ret):
                break
            if not np.isnan(ssm_ret[i-1]) and not np.isnan(ratio_arr[i]):
                ssm_ret[i] = ratio_arr[i] * ssm_ret[i-1]
            else:
                continue
    
    return ssm_ret


sigma_TM['SSM_retrieved'] = sigma_TM.groupby(['Code', 'Band'], group_keys=False).apply(
    lambda g: pd.Series(run_recursion_numpy(g['Moist_situ'].values, g['diff_lin'].values), index=g.index)
)


In [137]:
sigma_TM = sigma_TM.dropna()
display(sigma_TM.iloc[0:25])

,Code,Date_Band,mean,std,Date,Band,Name,Type,diff_mean,diff_lin,Moist_situ,SSM_retrieved
0,1553694,20170516_VH,-24.067930,1.918145,2017-05-16,VH,DB_SB2_C,Beets,1.608618,1.448311,19.1125,19.112500
1,1553694,20170522_VH,-22.459312,1.242794,2017-05-22,VH,DB_SB2_C,Beets,2.084634,1.616082,16.2500,30.887368
3,1553694,20170603_VH,-17.689444,1.524553,2017-06-03,VH,DB_SB2_C,Beets,3.346001,2.160728,27.5125,123.852392
4,1553694,20170609_VH,-14.343443,1.547611,2017-06-09,VH,DB_SB2_C,Beets,-1.950233,0.638229,15.9500,15.950000
5,1553694,20170615_VH,-16.293676,1.391763,2017-06-15,VH,DB_SB2_C,Beets,0.239801,1.056769,6.7625,16.855467
6,1553694,20170627_VH,-16.053875,1.619533,2017-06-27,VH,DB_SB2_C,Beets,1.319586,1.355060,12.0125,22.840173
7,1553694,20170703_VH,-14.734289,1.456309,2017-07-03,VH,DB_SB2_C,Beets,-1.610401,0.690176,27.4625,15.763741
8,1553694,20170709_VH,-16.344690,1.271146,2017-07-09,VH,DB_SB2_C,Beets,0.264630,1.062828,25.8500,25.850000
9,1553694,20170715_VH,-16.080060,1.031530,2017-07-15,VH,DB_SB2_C,Beets,-0.294951,0.934340,27.5750,24.152687
10,1553694,20170727_VH,-15.803813,1.147287,2017-07-27,VH,DB_SB2_C,Beets,2.074914,1.612469,30.3500,38.945461


Regression

In [138]:
from scipy import stats

results_regression = []

for (codice, banda, name,typ), gruppo in sigma_TM.groupby(['Code', 'Band', 'Name', 'Type']):        

    moist_situ = gruppo['Moist_situ'].values
    SSM_ret = gruppo['SSM_retrieved'].values

    if len(moist_situ) > 2:

        slope, intercept, r_value, p_value, std_err = stats.linregress(moist_situ, SSM_ret)        
        y_pred = slope * moist_situ + intercept
        
        residui = SSM_ret - y_pred
        
        rmsre = np.sqrt(np.mean(residui ** 2))
        
        results_regression.append({
            'Code': codice,
            'Name': name,
            'Band': banda,
            'Type': typ,
            'N_punti': len(moist_situ),
            'R_Pearson': r_value,
            'R_quadrato': r_value ** 2,
            'P_value': p_value,
            'Slope': slope,
            'Intercept': intercept,
            'RMSRE': rmsre
        })

df_statistiche = pd.DataFrame(results_regression)
df_statistiche.sort_values('R_quadrato', ascending=False, inplace=True)
display(df_statistiche)

,Code,Name,Band,Type,N_punti,R_Pearson,R_quadrato,P_value,Slope,Intercept,RMSRE
29,1936135,AKW_T_C,VV,Wheat,13,0.887881,0.788332,0.000051,1.331773,-3.157247,5.700049
28,1936135,AKW_T_C,VH,Wheat,13,0.823667,0.678427,0.000538,1.422751,-2.738849,8.090828
16,1841223,DB_T_C,VH,Wheat,13,0.706155,0.498656,0.006982,1.337336,-1.559548,9.283369
37,2081887,SB10_MA_C,VV,Corn,11,0.605170,0.366231,0.048521,1.346407,-3.579146,9.934170
36,2081887,SB10_MA_C,VH,Corn,11,0.559046,0.312533,0.073793,1.377382,-3.033560,11.457760
14,1824038,AKW_G2_C,VH,Grassland,11,0.493619,0.243659,0.122816,0.386187,12.188710,5.419662
17,1841223,DB_T_C,VV,Wheat,13,0.491942,0.242007,0.087716,1.468026,-0.204400,17.986639
15,1824038,AKW_G2_C,VV,Grassland,11,0.466421,0.217548,0.148138,0.329419,13.903975,4.976312
2,1576641,HF3_WT_C,VH,Wheat,10,0.435781,0.189905,0.208066,1.069179,10.397252,16.027927
19,1841224,DB_A_C,VV,Potatoes,14,0.416417,0.173403,0.138585,1.318389,0.213018,19.630708


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
from matplotlib.backends.backend_pdf import PdfPages


output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')
output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'Regression_Moisture4.pdf'

with PdfPages(pdf_filename) as pdf:

    for i, (_, row) in enumerate(df_statistiche.iterrows()):

        codice = row['Code']
        banda = row['Band']
        tipo = row['Type']
        
        # Estraiamo i dati temporali corrispondenti dal dataframe originale
        dati_gruppo = sigma_TM[(sigma_TM['Code'] == codice) & (sigma_TM['Band'] == banda)]
        
        x_in_situ = dati_gruppo['Moist_situ'].values
        y_calcolata = dati_gruppo['SSM_retrieved'].values
        
        # Eseguiamo il plot solo se ci sono abbastanza punti iniziali
        if len(x_in_situ) > 2:
            
            # --- CALCOLO DEGLI OUTLIER SUI DATI GREZZI ---
            # Usiamo pendenza e intercetta del primo ciclo (tutti i dati) per calcolare i residui
            y_pred_grezza = row['Slope'] * x_in_situ + row['Intercept']
            residui = y_calcolata - y_pred_grezza
            
            # Calcolo della soglia critica (3 * RMSRE)
            soglia = 3 * row['RMSRE']
            
            # Separazione Inlier / Outlier
            mask_outlier = np.abs(residui) > soglia
            mask_inlier = np.abs(residui) <= soglia
            n_outliers = np.sum(mask_outlier)
            
            x_puliti = x_in_situ[mask_inlier]
            y_puliti = y_calcolata[mask_inlier]
            
            # Procediamo al plot solo se restano abbastanza punti validi dopo la pulizia
            if len(x_puliti) > 2:
                # Inizializziamo la figura ad ogni iterazione per evitare sovrapposizioni
                plt.figure(figsize=(8, 5))
                
                # Ricalcoliamo la regressione SOLO per la retta del grafico (al netto degli outlier)
                slope_f, intercept_f, r_f, _, std_err = stats.linregress(x_puliti, y_puliti)
                r2_pulito = r_f ** 2
                
                # 1. Disegniamo i punti validi (Teal)
                sns.scatterplot(x=x_puliti, y=y_puliti, 
                                color='#1f77b4', s=60, label='Valid Data', alpha=0.8)
                
                # 2. Disegniamo gli outlier rimossi (Rosso con 'X') - Corretto 'marker' e 'label'
                if n_outliers > 0:
                    sns.scatterplot(x=x_in_situ[mask_outlier], y=y_calcolata[mask_outlier], 
                                    marker='x', color="#ca4828", s=60, label='Outliers', alpha=0.8)
                
                # 3. Disegniamo la retta di regressione PULITA
                x_linea = np.linspace(x_puliti.min(), x_puliti.max(), 100)
                y_linea = slope_f * x_linea + intercept_f
                sns.lineplot(x=x_linea, y=y_linea, color="#ca4828", 
                             label=f'Cleaned Fit\n$R^2$ w/out outlier = {r2_pulito:.3f}', alpha=0.8)
                
                plt.fill_between(x_linea, 
                                 y_linea - std_err, 
                                 y_linea + std_err, 
                                 color="#ca4828", alpha=0.15, label='1 Std. Dev.')


                # Linea di riferimento ideale 1:1
                limiti = [min(x_in_situ.min(), y_calcolata.min()), max(x_in_situ.max(), y_calcolata.max())]
                plt.plot(limiti, limiti, color='gray', linestyle=':', alpha=0.5, label='Ideale 1:1')
                
                # Titolo e formattazione assi
                plt.title(f"{codice} - {banda} - {tipo}\n($R^2$ raw: {row['R_quadrato']:.3f})",
                          fontsize=11, fontweight='bold')
                plt.xlabel("SSM In Situ ($m^3/m^3$)", fontsize=10)
                plt.ylabel("SSM Retr ($m^3/m^3$)", fontsize=10)
                plt.legend(loc='upper left', fontsize=9)
                plt.grid(True, linestyle='--', alpha=0.5)
                
                plt.tight_layout()
                
                # CRITICO: Prima salviamo la pagina nel PDF, poi eventualmente mostriamo a schermo
                pdf.savefig()
                # plt.show() # Decommentare se si desidera visualizzarli anche a schermo durante l'esecuzione
                plt.close()

       
print(f"Graphs saved: {pdf_filename}")
display()


Graphs saved: /home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/Regression_Moisture4.pdf


In [140]:
df_rainfall = pd.read_parquet(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/rainfall_total.parquet')
df_rainfall.index = pd.to_datetime(df_rainfall.index, format='%Y-%m-%d')
prima_corrispondenza = next((d for d in df_rainfall.index if np.datetime64(d) == np.datetime64(sigma_TM['Date'].iloc[0])), None)

print(f"Data trovata: {prima_corrispondenza}")
df_filtered = df_rainfall.loc[prima_corrispondenza:]
df_weekly = df_filtered.resample('6D', label='left').sum()


Data trovata: 2017-05-16 00:00:00


In [141]:
from matplotlib.backends.backend_pdf import PdfPages
# 1. Trasformazione del dizionario in DataFrame (Formato Lungo)
# Definisci il nome del file di uscita

sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date']).dt.tz_localize(None)

output_folder = app_path / 'Documents'
output_folder.mkdir(parents=True, exist_ok=True)

pdf_filename = output_folder / 'TS_diff_mean_Rain.pdf'

with PdfPages(pdf_filename) as pdf:
    unique_rois = sigma_TM['Code'].unique()

    for (roi), group in sigma_TM.groupby(['Code']):
        # 1. Prepariamo la figura e gli assi
        fig, ax1 = plt.subplots(figsize=(10, 6))
        
        # Filtriamo i dati ROI
        roi_data = sigma_TM[sigma_TM['Code'] == roi]
        
        
        # --- SICUREZZA DATE ---
        # Filtriamo le piogge per mostrare solo lo stesso periodo della coerenza
        min_date = roi_data['Date'].min()
        max_date = roi_data['Date'].max()
        rain_filtered = df_weekly[(df_weekly.index >= min_date) & (df_weekly.index <= max_date)]

        # --- ASSE 2: PIOGGIA (Sotto le linee) ---
        ax2 = ax1.twinx()
        # Usiamo zorder=1 per metterlo sullo sfondo
        ax2.bar(rain_filtered.index, rain_filtered.iloc[:,0], 
                color='skyblue', alpha=0.3, label='Rainfall (mm)', width=0.8, zorder=1)
        
        ax2.set_ylabel("Rainfall (mm)", color='tab:blue', fontsize=12, fontweight='bold')
        ax2.tick_params(axis='y', labelcolor='tab:blue')
        
        # Spazio sopra le barre per non coprire le linee (regola il moltiplicatore se serve)
        if not rain_filtered.empty and rain_filtered.iloc[:,0].max() > 0:
            ax2.set_ylim(0, rain_filtered.iloc[:,0].max() * 2.5)
        ax2.invert_yaxis() 


        ax1.set_zorder(ax2.get_zorder() + 1)
        ax1.patch.set_visible(False) # Rende ax1 trasparente per vedere ax2 sotto
        
        for band in roi_data['Band'].unique():
            band_data = roi_data[roi_data['Band'] == band]
            line, = ax1.plot(band_data['Date'], band_data['diff_mean'], 
                             marker='o', markersize=4, linestyle='-', linewidth=2, 
                             label=f'diff_mean {band}', zorder=3)
            
            


        ax1.set_ylabel("diff_mean", fontsize=12, fontweight='bold')
        ax1.set_xlabel("Data", fontsize=12)

        # --- LEGENDA E LAYOUT ---
        # Uniamo le labels di entrambi gli assi
        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        
        # Spostiamo la legenda leggermente più a destra e usiamo subplots_adjust
        ax1.legend(h1 + h2, l1 + l2, loc='upper left', bbox_to_anchor=(1.08, 1), borderaxespad=0.)

        plt.title(f"ROI: {roi} - diff_mean vs Rainfall", fontsize=14, pad=20)
        ax1.grid(True, linestyle='--', alpha=0.4)
        
        # Ruotiamo le date sull'asse X
        plt.setp(ax1.get_xticklabels(), rotation=45)
        
        # Invece di tight_layout puro, usiamo margin per far spazio alla legenda
        plt.subplots_adjust(right=0.85, bottom=0.15)
        
        pdf.savefig(fig)
        plt.close(fig)